In [2]:
import cv2
import time
import mediapipe as mp
from google.colab import files

VIDEO_PATH = "input_video.mp4"
OUTPUT_PATH = "output_comparison.mp4"

# 1. Open the input video
cap = cv2.VideoCapture(VIDEO_PATH)

# Get video properties to configure the VideoWriter
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
orig_fps = cap.get(cv2.CAP_PROP_FPS)

# Because we are putting them side-by-side, the output width is doubled
out_width = width * 2
out_height = height

# 2. Initialize the VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, orig_fps, (out_width, out_height))

# 3. Initialize both Detectors
haar_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
mp_face = mp.solutions.face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5)

print("Processing video and generating side-by-side comparison...")
print("This may take a few minutes depending on the video length.")

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Create separate copies for each detector
    frame_haar = frame.copy()
    frame_mp = frame.copy()

    # ==========================================
    # DETECTOR 1: HAAR CASCADE (Left Side)
    # ==========================================
    start_haar = time.time()
    gray = cv2.cvtColor(frame_haar, cv2.COLOR_BGR2GRAY)
    haar_faces = haar_cascade.detectMultiScale(gray, 1.3, 5)

    haar_time = time.time() - start_haar
    haar_fps = 1.0 / haar_time if haar_time > 0 else 0

    # Draw Green boxes
    for (x, y, w, h) in haar_faces:
        cv2.rectangle(frame_haar, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # Overlay Haar Stats
    cv2.putText(frame_haar, "HAAR CASCADE", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
    cv2.putText(frame_haar, f"Speed: {haar_fps:.1f} FPS", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame_haar, f"Faces Found: {len(haar_faces)}", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # ==========================================
    # DETECTOR 2: MEDIAPIPE (Right Side)
    # ==========================================
    start_mp = time.time()
    rgb = cv2.cvtColor(frame_mp, cv2.COLOR_BGR2RGB)
    mp_results = mp_face.process(rgb)

    mp_time = time.time() - start_mp
    mp_fps = 1.0 / mp_time if mp_time > 0 else 0

    mp_face_count = 0
    if mp_results.detections:
        mp_face_count = len(mp_results.detections)
        # Draw Blue boxes
        for det in mp_results.detections:
            bbox = det.location_data.relative_bounding_box
            x = int(bbox.xmin * width)
            y = int(bbox.ymin * height)
            w = int(bbox.width * width)
            h = int(bbox.height * height)
            cv2.rectangle(frame_mp, (x, y), (x+w, y+h), (255, 0, 0), 2)

    # Overlay MediaPipe Stats
    cv2.putText(frame_mp, "MEDIAPIPE (DEEP LEARNING)", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 3)
    cv2.putText(frame_mp, f"Speed: {mp_fps:.1f} FPS", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.putText(frame_mp, f"Faces Found: {mp_face_count}", (20, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    # ==========================================
    # STITCH AND SAVE
    # ==========================================
    # Combine frames side-by-side horizontally
    combined_frame = cv2.hconcat([frame_haar, frame_mp])

    # Write the stitched frame to the new video file
    out.write(combined_frame)

# Clean up
cap.release()
out.release()
mp_face.close()

print(f"\nDone! Processed {frame_count} frames.")
print(f"Video saved as: {OUTPUT_PATH}")

# Trigger browser download automatically
files.download(OUTPUT_PATH)

Processing video and generating side-by-side comparison...
This may take a few minutes depending on the video length.

Done! Processed 265 frames.
Video saved as: output_comparison.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>